# 1. Base Setup

## 1.1 Install packages

In [14]:
# %load_ext autoreload
# %autoreload 2

In [15]:
# !pip install kagglehub
# !pip install statsmodels
# !pip install xgboost
# !pip install catboost

## 1.2 Load all needed imports

In [16]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix, accuracy_score, f1_score
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV

from scipy.stats import randint, uniform
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [17]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd()
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [18]:
def data_cleaning(df: pd.DataFrame, predict: bool=False) -> tuple:
    """Clean raw data and split into model and demo DataFrames.

    Deduplicates rows, parses dates, renames misspelled columns, selects
    relevant columns, and splits into a model_df (paid invoices only)
    and demo_df (all invoices).

    Args:
        df: raw DataFrame from load_cashflow_data().

    Returns:
        A tuple (model_df, demo_df).
    """
    # to remove warning of SettingWithCopyWarning:
    df = df.copy()

    # drop duplicates and name stripping
    df = df.drop_duplicates()
    df.columns = df.columns.str.strip()

    if "cust_number" in df.columns:
        df["cust_number"] = df["cust_number"].astype(str)
    # Drop invalid invoice ids
    df = df.dropna(subset=["invoice_id"]).copy()
    df["invoice_id"] = pd.to_numeric(df["invoice_id"], errors="coerce")
    df = df[df["invoice_id"].notna()].copy()

    # Parse date columns
    df["due_in_date"] = pd.to_datetime(df["due_in_date"], format="%Y%m%d", errors="coerce")
    df["baseline_create_date"] = pd.to_datetime(df["baseline_create_date"], format="%Y%m%d", errors="coerce")
    df["clear_date"] = pd.to_datetime(df["clear_date"], errors="coerce")
    # Cast IDs
    df["doc_id"] = df["doc_id"].astype("int64")

    # Standardize categorical columns
    cat_cols = ["business_code", "name_customer", "invoice_currency", "document type", "cust_payment_terms"]
    for c in cat_cols:
        df[c] = df[c].astype(str)

    # Select and rename columns
    df = df[['doc_id','cust_number', 'buisness_year', 'due_in_date', 'invoice_currency',
             'document type', 'total_open_amount', 'baseline_create_date',
             'cust_payment_terms', 'clear_date']]

    df.rename(columns={
        'buisness_year': 'business_year',
        'clear_date': 'invoice_paid',
        'document type': 'document_type',
        'baseline_create_date': 'invoice_sent',
    }, inplace=True)

    # sort values
    df = df.sort_values("invoice_sent").reset_index(drop=True)
    # data conversion
    df["business_year"] = df["business_year"].round().astype("int64")

    # converting cad to usd
    cad_to_usd_by_year = {
        2018 : 0.771,
        2019 : 0.754,
        2020 : 0.745
    }
    def convert_cad_to_usd_amount(row):
        if row['invoice_currency'].strip().upper() == "USD":
            return row["total_open_amount"]
        elif row['invoice_currency'].strip().upper() == "CAD":
            year = int(row['business_year'])
            rate = cad_to_usd_by_year.get(year,0.75)
            return row["total_open_amount"] * rate
        else:
            return row["total_open_amount"]

    df["total_open_amount"] = df.apply(convert_cad_to_usd_amount,axis=1)

    if not predict:
        df = df[df["invoice_paid"].notnull()]
    # Save processed frames
    base_dir = Path.cwd()
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    # if not predict:
        # df.to_csv(raw_data_dir / "df.csv", index=False)
    #demo_df.to_csv(raw_data_dir / "demo_df.csv", index=False)

    #print(f"Saved df ({len(df)} rows) and demo_df ({len(demo_df)} rows)")
    return df

In [19]:
def engineer_features(snapshot: pd.DataFrame, df_full: pd.DataFrame,
                      current_date) -> pd.DataFrame:
    """Engineer all features for a given reference date.

    Adds invoice timing features, customer behaviour features (from full
    history), and amount-based features to the snapshot.

    Args:
        snapshot: DataFrame of open invoices at current_date.
        df_full: full invoice DataFrame (for historical calculations).
        current_date: the reference date for the snapshot.

    Returns:
        The snapshot DataFrame with all engineered features added.
    """
    # A) Invoice timing features
    open_invoice = snapshot["invoice_paid"].isna() | (snapshot["invoice_paid"] > current_date)
    snapshot["invoice_age_days"] = np.where(
        open_invoice,
        (current_date - snapshot["invoice_sent"]).dt.days,
        np.nan,
    )
    snapshot["days_until_due"] = np.where(
        open_invoice,
        (snapshot["due_in_date"] - current_date).dt.days,
        np.nan,
    )
    snapshot["pay_terms_days"] = (snapshot["due_in_date"] - snapshot["invoice_sent"]).dt.days
    snapshot["invoice_month"] = snapshot["invoice_sent"].dt.month
    snapshot["due_month"] = snapshot["due_in_date"].dt.month
    snapshot["days_past_due"] = (current_date - snapshot["due_in_date"]).dt.days

    # B) Customer behaviour features
    historical = df_full[df_full["invoice_paid"] <= current_date].copy()
    historical["delay"] = (historical["invoice_paid"] - historical["due_in_date"]).dt.days.clip(lower=0)
    avg_delay = historical.groupby("cust_number")["delay"].mean().rename("customer_avg_delay")

    historical["is_late"] = (historical["invoice_paid"] > historical["due_in_date"]).astype(int)
    late_ratio = historical.groupby("cust_number")["is_late"].mean().rename("late_payment_ratio")

    before_current = df_full[df_full["invoice_sent"] < current_date]
    prev_counts = before_current.groupby("cust_number").size().rename("prev_transaction_count")

    last_invoice = (
        before_current.groupby("cust_number")["invoice_sent"]
        .max()
        .rename("last_invoice_date")
    )

    customer_features = pd.concat([avg_delay, late_ratio, prev_counts, last_invoice], axis=1)
    snapshot = snapshot.merge(customer_features, on="cust_number", how="left")

    snapshot["customer_risk_score"] = (
        0.7 * snapshot["late_payment_ratio"].fillna(0)
        + 0.3 * snapshot["customer_avg_delay"].fillna(0)
    )

    snapshot["days_since_last_invoice"] = (current_date - snapshot["last_invoice_date"]).dt.days
    snapshot.drop(columns=["last_invoice_date"], inplace=True)
    snapshot["prev_transaction_count"] = snapshot["prev_transaction_count"].fillna(0).astype(int)

    # C) Invoice characteristics
    snapshot["invoice_amount"] = snapshot["total_open_amount"]
    snapshot["invoice_amount_log"] = np.log1p(snapshot["invoice_amount"].clip(lower=0))
    snapshot["open_amount"] = snapshot["total_open_amount"]

    size_bins_fixed = [-np.inf, 10000, 100000, np.inf]
    size_labels = ["small", "medium", "large"]
    snapshot["invoice_size_cat"] = pd.cut(
        snapshot["invoice_amount"], bins=size_bins_fixed, labels=size_labels,
    )

    q1, q2 = df_full["total_open_amount"].quantile([0.33, 0.66])
    size_bins_quant = [-np.inf, q1, q2, np.inf]
    snapshot["invoice_size_cat_q"] = pd.cut(
        snapshot["invoice_amount"], bins=size_bins_quant, labels=size_labels,
    )

    return snapshot

In [20]:
def build_sliding_window_snapshots(df: pd.DataFrame) -> pd.DataFrame:
    """Build augmented dataset by sliding a weekly window over invoice history.

    For each weekly snapshot, selects all invoices sent but not yet paid,
    engineers features, and computes the target 'week_bucket' (1-7).

    Args:
        df: cleaned DataFrame with 'invoice_sent' and 'invoice_paid' columns.

    Returns:
        Concatenated DataFrame of all weekly snapshots with target column.
    """
    all_snapshots = []

    start_date = df["invoice_sent"].min()
    end_date = df["invoice_paid"].max() - pd.Timedelta(weeks=6)
    stride = pd.Timedelta(weeks=1)

    current = start_date
    while current <= end_date:
        snapshot = df[
            (df["invoice_sent"] <= current) &
            (df["invoice_paid"] > current)
        ].copy()

        snapshot = engineer_features(snapshot, df, current)

        snapshot["reference_date"] = current
        snapshot["days_to_payment"] = (snapshot["invoice_paid"] - current).dt.days
        snapshot["week_bucket"] = np.ceil(snapshot["days_to_payment"] / 7).astype(int)
        snapshot["week_bucket"] = snapshot["week_bucket"].clip(lower=1)
        snapshot.loc[snapshot["week_bucket"] > 7, "week_bucket"] = 7

        all_snapshots.append(snapshot)
        current += stride

    big_df = pd.concat(all_snapshots, ignore_index=True)
    print(f"Original rows: {len(df)}")
    print(f"Augmented rows: {len(big_df)}")
    print(big_df["week_bucket"].value_counts().sort_index())

    return big_df

In [21]:
NUMERIC_FEATURES = [
    "business_year", "invoice_age_days", "days_until_due", "pay_terms_days",
    "invoice_month", "due_month", "days_past_due", "customer_avg_delay",
    "late_payment_ratio", "prev_transaction_count", "days_since_last_invoice",
    "customer_risk_score", "invoice_amount", "invoice_amount_log",
]

CATEGORICAL_FEATURES = [
    "invoice_currency", "document_type", "invoice_size_cat", "invoice_size_cat_q"
]


def preprocess(df: pd.DataFrame, inference: bool = False) -> tuple:
    """Preprocess a DataFrame for model training or inference.

    Imputes missing values in customer history features and splits into
    feature matrix and (optionally) target vector. Identifier columns,
    date columns, and leaky columns are excluded.

    Args:
        df: pandas DataFrame containing invoice-level records.
        inference: if True, skips dropping rows without a target and
                   returns y as None. Set this when predicting on new
                   invoices that have no 'week_bucket' column yet.

    Returns:
        A tuple (X, y) where X is a DataFrame of feature columns and y is
        a Series of 'week_bucket' target labels (or None during inference).
    """
    df = df.copy()

    if not inference:
        df = df.dropna(subset=["week_bucket"])

    df["customer_avg_delay"] = df["customer_avg_delay"].fillna(0)
    df["late_payment_ratio"] = df["late_payment_ratio"].fillna(0)
    df["days_since_last_invoice"] = df["days_since_last_invoice"].fillna(-1)

    drop_cols = [
        "doc_id","cust_number", "due_in_date", "invoice_sent", "total_open_amount",
        "open_amount", "reference_date", "week_bucket", "days_to_payment",
        "invoice_paid", "cust_payment_terms"
    ]
    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols]
    y = df["week_bucket"] if "week_bucket" in df.columns and not inference else None

    return X, y

In [22]:
hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)
cutoff_date = big_df["invoice_sent"].quantile(0.9)

train_df = big_df[big_df["reference_date"] <= cutoff_date]
test_df = big_df[big_df["reference_date"] > cutoff_date]

X_train, y_train = preprocess(train_df)
X_test, y_test = preprocess(test_df)

Loading local file: /Users/anton/code/DERNTOAN/cf_copilot/notebooks/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64


In [23]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [24]:
from cf_copilot.cashflow_prediction.registry import WEEK_CLASSES

def forecast_mape_scorer(estimator, X, y):
    """
    Custom scorer matching evaluate_forecast_holdout MAPE logic.
    Needs access to total_open_amount in the original dataframe.
    """
    probas = estimator.predict_proba(X)
    model_classes = np.array(estimator.named_steps["classifier"].classes_)

    if np.array_equal(np.sort(model_classes), np.array([0, 1, 2, 3, 4, 5, 6])):
        class_to_index = {int(c) + 1: i for i, c in enumerate(model_classes)}
    else:
        class_to_index = {int(c): i for i, c in enumerate(model_classes)}

    # You'll need amounts accessible — either pass through the pipeline
    # or store them externally. This assumes X still has the column:
    amounts = X["total_open_amount"].values

    forecast_per_week = {}
    actual_per_week = {}
    for w in WEEK_CLASSES:
        idx = class_to_index.get(int(w))
        p_w = probas[:, idx] if idx is not None else np.zeros(len(X))
        forecast_per_week[w] = (amounts * p_w).sum()
        actual_per_week[w] = amounts[y == int(w)].sum()

    mape_values = []
    for w in WEEK_CLASSES:
        actual = actual_per_week[w]
        if actual != 0:
            mape_values.append(abs((actual - forecast_per_week[w]) / actual))

    return -np.mean(mape_values) if mape_values else 0.0

In [25]:
from xgboost import XGBClassifier
from sklearn.feature_selection import VarianceThreshold

classifier = XGBClassifier(
    colsample_bytree=0.727,
    gamma=0.059,
    learning_rate=0.025,
    max_depth=7,
    min_child_weight=2,
    n_estimators=352,
    reg_alpha=0.587,
    reg_lambda=1.931,
    subsample=0.882,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

XGB_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("variance", VarianceThreshold(threshold=0.05)),
    ("classifier", classifier),
])

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
import datetime

# This is XGBoost specific
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

tscv = TimeSeriesSplit(n_splits=5)

param_distributions = {
    # Core boosting capacity
    "classifier__n_estimators": randint(200, 501),
    "classifier__learning_rate": uniform(0.01, 0.04),

    # Tree complexity
    "classifier__max_depth": randint(4, 10),
    "classifier__min_child_weight": randint(1, 7),

    # Split regularization
    "classifier__gamma": uniform(0.0, 0.2),

    # Sampling
    "classifier__subsample": uniform(0.7, 0.25),
    "classifier__colsample_bytree": uniform(0.5, 0.4),

    # Regularization
    "classifier__reg_alpha": uniform(0.1, 1.2),
    "classifier__reg_lambda": uniform(1.0, 2.5),

    # Categorical imputer
    "preprocessor__cat__imputer__strategy": ["constant", "most_frequent"],
    "preprocessor__cat__imputer__fill_value": [-1, -2, 0],

    # Categorical encoder
    "preprocessor__cat__encoder__unknown_value": [-1, -2],
}

random_search = RandomizedSearchCV(
    estimator=XGB_pipeline,
    param_distributions=param_distributions,
    n_iter=500,
    cv=tscv,
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=1,
    random_state=42,
    error_score=np.nan,
    return_train_score=True,
)

y_search_xgb = y_train_enc
y_final_test_xgb = y_test_enc

random_search.fit(X_train, y_search_xgb)

best_model = random_search.best_estimator_

Fitting 5 folds for each of 1 candidates, totalling 5 fits


In [ ]:
print("DONE WITH THE FITTING")

In [28]:
best_model = random_search.best_estimator_

y_proba = best_model.predict_proba(X_test)
y_pred = best_model.predict(X_test)

clf_classes = np.array(best_model.named_steps["classifier"].classes_)

search_classes = np.array(sorted(pd.Series(y_train_enc).unique()))
test_classes = np.array(sorted(pd.Series(y_test_enc).unique()))

cv_results = pd.DataFrame(random_search.cv_results_)
n_failed = cv_results["mean_test_score"].isna().sum()
print(f"Failed parameter sets: {n_failed} / {len(cv_results)}")

missing_in_train = set(test_classes) - set(clf_classes)
print("Classes in final test but not seen during search:", missing_in_train)

if missing_in_train:
    raise ValueError(
        f"Final test contains unseen classes: {missing_in_train}. "
        "The model cannot evaluate those correctly."
    )

if y_proba.shape[1] != len(clf_classes):
    raise ValueError(
        f"Mismatch between probability columns ({y_proba.shape[1]}) "
        f"and classifier classes ({len(clf_classes)})."
    )

print("Best CV log_loss:", -random_search.best_score_)
print("Best params:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

final_log_loss = log_loss(y_test_enc, y_proba, labels=clf_classes)
final_accuracy = accuracy_score(y_test_enc, y_pred)
final_f1_macro = f1_score(y_test_enc, y_pred, average="macro")

print("\nFinal holdout performance")
print(f"log_loss  : {final_log_loss:.6f}")
print(f"accuracy  : {final_accuracy:.6f}")
print(f"f1_macro  : {final_f1_macro:.6f}")

print("\nClass checks")
print("Search classes:", list(search_classes))
print("Test classes  :", list(test_classes))
print("Model classes :", list(clf_classes))

best_idx = random_search.best_index_
best_train_score = random_search.cv_results_["mean_train_score"][best_idx]
best_valid_score = random_search.cv_results_["mean_test_score"][best_idx]

print("\nBest CV split scores")
print(f"Best mean train neg_log_loss: {best_train_score:.6f}")
print(f"Best mean valid neg_log_loss: {best_valid_score:.6f}")
print(f"Best mean train log_loss    : {-best_train_score:.6f}")
print(f"Best mean valid log_loss    : {-best_valid_score:.6f}")

display(
    cv_results.sort_values("rank_test_score")[
        ["rank_test_score", "mean_test_score", "mean_train_score", "params"]
    ].head(5)
)

Failed parameter sets: 0 / 1
Classes in final test but not seen during search: set()
Best CV log_loss: 0.7178946843105096
Best params:
  classifier__colsample_bytree: 0.649816047538945
  classifier__gamma: 0.19014286128198324
  classifier__learning_rate: 0.03927975767245621
  classifier__max_depth: 8
  classifier__min_child_weight: 5
  classifier__n_estimators: 302
  classifier__reg_alpha: 0.6349993034243093
  classifier__reg_lambda: 1.2499372895450072
  classifier__subsample: 0.8148122229914667

Final holdout performance
log_loss  : 0.816771
accuracy  : 0.724432
f1_macro  : 0.554479

Class checks
Search classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Test classes  : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Model classes : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Best CV split scores
Best mean train neg_log_loss: -0.448100
Best mean

,rank_test_score,mean_test_score,mean_train_score,params
0,1,-0.717895,-0.4481,{'classifier__colsample_bytree': 0.64981604753...


In [29]:
from cf_copilot.cashflow_prediction.evaluation import build_actual_weekly_cf, compare_forecast_vs_actual, compute_forecast_metrics
from cf_copilot.cashflow_prediction.registry import WEEK_CLASSES
from colorama import Fore, Style
from cf_copilot.ml_logic.encoders import preprocess


def evaluate_forecast_holdout(
    model: Pipeline,
    test_df: pd.DataFrame,
    verbose: bool = True
) -> tuple[dict, dict]:
    """
    Evaluate forecast quality on the holdout set by reference_date.

    Args:
        model: fitted sklearn Pipeline
        test_df: holdout snapshot dataframe
        verbose: whether to print evaluation output

    Returns:
        forecast_metrics: scalar metrics to merge into MLflow metrics
        forecast_summary: JSON-serializable artifact with per-reference-date detail
    """
    if verbose:
        print(Fore.BLUE + f"\nEvaluating forecast on {len(test_df)} rows..." + Style.RESET_ALL)

    if model is None:
        if verbose:
            print("❌ No model to evaluate forecast")
        return {}, {}

    if len(test_df) == 0:
        if verbose:
            print("❌ No holdout rows available for forecast evaluation")
        return {}, {}

    per_reference_results = []
    if hasattr(model, "named_steps") and "classifier" in model.named_steps:
        model_classes = np.array(model.named_steps["classifier"].classes_)
    else:
        model_classes = np.array(model.classes_)

    # Support both label schemes:
    # standard models: 1..7
    # XGBoost: 0..6
    if np.array_equal(np.sort(model_classes), np.array([0, 1, 2, 3, 4, 5, 6])):
        class_to_index = {int(c) + 1: i for i, c in enumerate(model_classes)}
    else:
        class_to_index = {int(c): i for i, c in enumerate(model_classes)}

    for reference_date, snapshot_df in test_df.groupby("reference_date"):
        snapshot_df = snapshot_df.copy()
        if len(snapshot_df) == 0:
            continue

        X_snapshot, _ = preprocess(snapshot_df)
        probas = model.predict_proba(X_snapshot)

        pred_cash_df = snapshot_df.copy()

        for w in WEEK_CLASSES:
            pred_cash_df[f"p_{w}"] = (
                probas[:, class_to_index[int(w)]]
                if int(w) in class_to_index
                else 0.0
            )

        weekly_forecast_df = pd.DataFrame([
            {
                "week_bucket": int(w),
                "forecast_cash": round(
                    float((pred_cash_df["total_open_amount"] * pred_cash_df[f"p_{w}"]).sum()),
                    2,
                ),
            }
            for w in WEEK_CLASSES
        ])

        total_invoice_amount = round(float(pred_cash_df["total_open_amount"].sum()), 2)
        total_expected_cash = round(float(weekly_forecast_df["forecast_cash"].sum()), 2)

        if total_expected_cash - total_invoice_amount > 0:
            print("Expected cash exceeds total invoice amount. This should not happen.")

        actual_weekly_df = build_actual_weekly_cf(
            invoices_df=snapshot_df,
            reference_date=pd.Timestamp(reference_date),
        )

        comparison_df = compare_forecast_vs_actual(
            weekly_forecast_df=weekly_forecast_df,
            actual_weekly_df=actual_weekly_df,
        )

        snapshot_metrics = compute_forecast_metrics(comparison_df)

        per_reference_results.append({
            "reference_date": str(pd.Timestamp(reference_date).date()),
            "forecast_check": {
                "total_invoice_amount": total_invoice_amount,
                "total_expected_cash": total_expected_cash,
            },
            "forecast_metrics": {
                "forecast_mae_weekly": float(snapshot_metrics["MAE (weekly)"]),
                "forecast_mape_weekly": (
                    float(snapshot_metrics["MAPE (weekly)"])
                    if pd.notna(snapshot_metrics["MAPE (weekly)"])
                    else None
                ),
                "total_actual_cf": float(snapshot_metrics["Total actual cf"]),
                "total_forecast_cf": float(snapshot_metrics["Total forecast cf"]),
                "total_cf_difference": float(snapshot_metrics["Total cf difference"]),
            },
        })

    if not per_reference_results:
        if verbose:
            print("❌ No per-reference-date forecast results available")
        return {}, {}

    mae_values = [r["forecast_metrics"]["forecast_mae_weekly"] for r in per_reference_results]
    mape_values = [
        r["forecast_metrics"]["forecast_mape_weekly"]
        for r in per_reference_results
        if r["forecast_metrics"]["forecast_mape_weekly"] is not None
    ]
    total_actual_values = [r["forecast_metrics"]["total_actual_cf"] for r in per_reference_results]
    total_forecast_values = [r["forecast_metrics"]["total_forecast_cf"] for r in per_reference_results]
    total_diff_values = [r["forecast_metrics"]["total_cf_difference"] for r in per_reference_results]

    forecast_metrics = {
        "forecast_mae_weekly": float(np.mean(mae_values)),
        "forecast_mape_weekly": float(np.mean(mape_values)) if mape_values else np.nan,
        "forecast_total_actual_cf": float(np.mean(total_actual_values)),
        "forecast_total_forecast_cf": float(np.mean(total_forecast_values)),
        "forecast_total_cf_difference": float(np.mean(total_diff_values)),
    }

    forecast_summary = {
        "per_reference_date": per_reference_results,
        "aggregate": {
            "forecast_mae_weekly": forecast_metrics["forecast_mae_weekly"],
            "forecast_mape_weekly": (
                forecast_metrics["forecast_mape_weekly"]
                if pd.notna(forecast_metrics["forecast_mape_weekly"])
                else None
            ),
            "forecast_total_actual_cf": forecast_metrics["forecast_total_actual_cf"],
            "forecast_total_forecast_cf": forecast_metrics["forecast_total_forecast_cf"],
            "forecast_total_cf_difference": forecast_metrics["forecast_total_cf_difference"]
        },
    }

    if verbose:
        print(f"✅ Forecast MAE weekly: {forecast_metrics['forecast_mae_weekly']:.2f}")

        if pd.notna(forecast_metrics["forecast_mape_weekly"]):
            print(f"✅ Forecast MAPE weekly: {forecast_metrics['forecast_mape_weekly']:.4f}")
        else:
            print("✅ Forecast MAPE weekly: None")

        print(f"✅ Forecast total actual cf: {forecast_metrics['forecast_total_actual_cf']:.2f}")
        print(f"✅ Forecast total forecast cf: {forecast_metrics['forecast_total_forecast_cf']:.2f}")
        print(f"✅ Forecast total cf difference: {forecast_metrics['forecast_total_cf_difference']:.2f}")

    return forecast_metrics, forecast_summary

ImportError: cannot import name 'sharpen_probabilities' from 'cf_copilot.cashflow_prediction.registry' (/Users/anton/code/DERNTOAN/cf_copilot/cf_copilot/cashflow_prediction/registry.py)

In [ ]:
import datetime

forecast_metrics, forecast_summary = evaluate_forecast_holdout(
    model=best_model,
    test_df=test_df,
    verbose=True,
)

print("\nAggregate forecast metrics")
for k, v in forecast_metrics.items():
    print(f"{k}: {v}")

pd.DataFrame([forecast_metrics])

print("END:", datetime.datetime.now())


Evaluating forecast on 11391 rows...
✅ Forecast MAE weekly: 494200.53
✅ Forecast MAPE weekly: 0.3392
✅ Forecast total actual cf: 24527177.14
✅ Forecast total forecast cf: 24238526.52
✅ Forecast total cf difference: -288650.62

Aggregate forecast metrics
forecast_mae_weekly: 494200.53153846157
forecast_mape_weekly: 0.3392307692307692
forecast_total_actual_cf: 24527177.14076923
forecast_total_forecast_cf: 24238526.517692305
forecast_total_cf_difference: -288650.6230769231
END: 2026-03-25 22:00:53.295449
